In [22]:
import joblib
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Định nghĩa đường dẫn root & output
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW_PATH = BASE_DIR / 'data' / 'raw' / 'gas_turbine_data.csv'  # chỉnh cho từng máy của từng thành viên

OUTPUT_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Đọc dữ liệu và loại bỏ bản ghi trùng lặp
df_raw = pd.read_csv(DATA_RAW_PATH)
initial_shape = df_raw.shape
df = df_raw.drop_duplicates().copy()
print(f"Đã xoá {initial_shape[0] - df.shape[0]} dòng trùng lặp. Kích thước mới: {df.shape}")


train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

# Copy để tránh warning SettingWithCopyWarning của Pandas
train_df = train_df.copy()
test_df = test_df.copy()

# 4. Ngưỡng capping p99 của CO chỉ học từ Train, áp dụng cho cả Train và Test
co_p99_threshold = train_df['CO'].quantile(0.99)
print(f"Giới hạn trên cho CO (99th percentile, chỉ tính trên Train): {co_p99_threshold}")

train_df['CO_clipped'] = train_df['CO'].clip(upper=co_p99_threshold)
test_df['CO_clipped'] = test_df['CO'].clip(upper=co_p99_threshold)

# 5. Scaler chỉ fit trên 9 đặc trưng đầu vào của Train, Test chỉ transform
# Xác định danh sách các biến đầu vào (Features) và đầu ra (Targets)
feature_cols = ['AT', 'AP', 'AH', 'AFDP', 'GTEP', 'TIT', 'TAT', 'TEY', 'CDP']
target_cols = ['CO', 'NOX']

scaler = StandardScaler()

train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print("Ví dụ dữ liệu Train sau khi StandardScaler (fit trên Train):")
print(train_df[feature_cols].head())

# 6. Lưu file processed và lưu artifacts (Scaler + Capping Threshold)
train_df.to_csv(OUTPUT_DIR / 'train_clean.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test_clean.csv', index=False)

# Lưu Scaler & Threshold ra thư mục models/ để tái lập suy luận sau này
joblib.dump(scaler, MODELS_DIR / 'scaler.joblib')
joblib.dump({'co_p99_threshold': co_p99_threshold}, MODELS_DIR / 'preprocessing_meta.joblib')

print(f"[THÀNH CÔNG] Đã lưu train_clean.csv ({len(train_df)} dòng) & test_clean.csv ({len(test_df)} dòng)")
print(f"[THÀNH CÔNG] Ngưỡng p99 CO tính từ Train: {co_p99_threshold:.4f} mg/m³")
print(f"[THÀNH CÔNG] Đã đóng gói scaler.joblib tại {MODELS_DIR}")

Đã xoá 7 dòng trùng lặp. Kích thước mới: (36726, 12)
Giới hạn trên cho CO (99th percentile, chỉ tính trên Train): 10.953090000000005
Ví dụ dữ liệu Train sau khi StandardScaler (fit trên Train):
             AT        AP        AH      AFDP      GTEP       TIT       TAT  \
13590  0.389520 -0.351710 -0.261321 -0.518025 -1.401522 -1.452927  0.548927   
10711  0.996069 -0.678402  0.636089  0.286516 -0.003954  0.473862  0.560622   
7012  -0.559029  0.628369  0.857849  0.316518 -0.356858  0.171732  0.554775   
31281  0.044300  0.037210 -0.615154 -1.293207 -1.011946 -1.464328  0.592784   
25065  1.353772 -0.678402 -1.193167  1.057963  1.428381  1.049619 -0.611827   

            TEY       CDP  
13590 -1.570197 -1.450926  
10711 -0.041634  0.115957  
7012  -0.058931 -0.190807  
31281 -1.235784 -1.210291  
25065  1.114718  1.207998  
[THÀNH CÔNG] Đã lưu train_clean.csv (25708 dòng) & test_clean.csv (11018 dòng)
[THÀNH CÔNG] Ngưỡng p99 CO tính từ Train: 10.9531 mg/m³
[THÀNH CÔNG] Đã đóng gói sca

In [23]:
# xử lý outlier (mọi ngưỡng chỉ tính trên Train)
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
processed_dir = BASE_DIR / 'data' / 'processed'
models_dir = BASE_DIR / 'models'
train_file = processed_dir / 'train_clean.csv'
train_df = pd.read_csv(train_file)

# Đưa 9 đặc trưng về đơn vị gốc (scaler đã fit trên Train) để phân tích outlier
feature_cols = ['AT', 'AP', 'AH', 'AFDP', 'GTEP', 'TIT', 'TAT', 'TEY', 'CDP']
scaler = joblib.load(models_dir / 'scaler.joblib')
df = train_df.copy()  # phân tích outlier chỉ trên Train
df[feature_cols] = scaler.inverse_transform(df[feature_cols])

print(f"Đã tải thành công tệp: {train_file}")
print(f"Kích thước ban đầu (Train): {df.shape[0]} dòng, {df.shape[1]} cột")

print("\n=== PHÂN TÍCH TỶ LỆ OUTLIER THEO PHƯƠNG PHÁP IQR (TẬP TRAIN) ===")
outlier_summary = []

for col in df.columns:
    if col == 'year':
        continue
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    num_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    pct = (num_outliers / len(df)) * 100
    
    outlier_summary.append({
        'Biến': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Số Outlier': num_outliers,
        'Tỷ lệ (%)': round(pct, 2)
    })

print(pd.DataFrame(outlier_summary).to_string(index=False))


# Ngưỡng p99 đã được học từ Train ở bước tiền xử lý, đọc lại từ artifacts
p99_co = joblib.load(models_dir / 'preprocessing_meta.joblib')['co_p99_threshold']
print(f"\nNgưỡng Capping bách phân vị 99 (p99) của CO (chỉ tính trên Train): {p99_co:.4f} mg/m3")

print(f"Giá trị Max cột CO gốc (Train): {df['CO'].max():.4f}")
print(f"Giá trị Max cột CO_clipped sau xử lý (Train): {df['CO_clipped'].max():.4f}")

Đã tải thành công tệp: D:\Hoctap\BTL - Machine Learning\BTL---Machine-Learning---co3117-\data\processed\train_clean.csv
Kích thước ban đầu (Train): 25708 dòng, 13 cột

=== PHÂN TÍCH TỶ LỆ OUTLIER THEO PHƯƠNG PHÁP IQR (TẬP TRAIN) ===
      Biến      Q1      Q3   IQR  Số Outlier  Tỷ lệ (%)
        AT   11.74   23.66 11.92           1       0.00
        AP 1008.80 1017.00  8.20         404       1.57
        AH   68.31   89.37 21.05         101       0.39
      AFDP    3.35    4.38  1.03         366       1.42
      GTEP   23.12   29.03  5.91           6       0.02
       TIT 1071.70 1096.90 25.20         215       0.84
       TAT  544.76  550.04  5.28        3490      13.58
       TEY  124.33  143.82 19.49          19       0.07
       CDP   11.43   12.83  1.40           7       0.03
        CO    1.19    2.86  1.67        1863       7.25
       NOX   57.19   71.59 14.41         689       2.68
CO_clipped    1.19    2.86  1.67        1863       7.25

Ngưỡng Capping bách phân vị 99 (p99) c